# Stage 4, tier 2: BioLangFusion Phase B, one encoder swapped at a time

Research plan item 1, Phase B (Gantt 24; Overleaf Section 12.2, item 1). Phase A reproduced BioLangFusion with its own three encoders. Phase B keeps everything else fixed (the splits, the 999-nucleotide rule, the codon grid, the three fusion heads, the TextCNN head, the training recipe, the evaluation) and replaces one encoder at a time. The question is whether a different pretrained encoder improves a published fusion system, and the pattern of gains across swaps is the plan's direct test of H2: does the complementarity Stage 1 measured on frozen pooled vectors predict which swaps help a trained fusion model?

**The swaps.** Every Stage 1 encoder that the codon grid can place, one at a time in its own modality's slot:

| Slot | BioLangFusion's encoder | Swapped in | How it reaches the grid |
| --- | --- | --- | --- |
| DNA | Nucleotide Transformer v2 100M | NT 500M human-ref | 6-mers, the same transposed convolution |
| DNA | | HyenaDNA large | one token per nucleotide, averaged in threes |
| RNA | RNA-FM | RiNALMo 150M | one token per nucleotide, averaged in threes |
| RNA | | mRNA-FM, CaLM | one token per codon, as they are |
| Protein | ESM-2 8M | ESM-2 35M, ESM-2 150M, ProtBERT | one token per residue, as they are |

The plan names CaLM, mRNA-FM, ProtBERT, ESM-2 150M and the alternative DNA encoders. RiNALMo and ESM-2 35M are added because Stage 1 found them redundant with the encoders they replace (RNA-FM with RiNALMo, CKA 0.81; the ESM-2 family 0.55 to 0.74), so H2 predicts they change little, and that prediction is as testable as the others. **DNABERT-2 waits**: its byte-pair tokens cover a variable number of nucleotides, so placing it on the grid needs an adapter of its own, designed once and held fixed (Overleaf Section 11.1).

**All four benchmarks**, one seed each, compared on the same test rows with the original encoders' runs from Phase A (and, for fungal, Stage 3 step 3). The plan names Fungal, E. coli, mRNA stability and Ab1, with CoV-Vac if feasible; Ab1 is not public, and the other four are here.

**mRNA stability is the expensive one** and runs last. It is eight times fungal, and a swapped encoder's token cache for it runs from 7 to 36 GB apiece, so the default there is three swaps rather than eight: mRNA-FM and CaLM in the RNA slot, and ESM-2 150M in the protein slot. That is not a budget cut dressed up as a choice. The mRFP control (research plan item 5) found mRNA-FM carrying by far the most synonymous-codon information of any encoder, +0.18 held-out $R^2$ above everything GC3, codon usage and folding explain, against +0.08 for the next; and the encoder BioLangFusion actually fuses, RNA-FM, carries the least. Putting mRNA-FM into the RNA slot on the benchmark whose labels are most codon-driven is the sharpest test this project has of whether what the mRFP control measured transfers to a trained fusion model. CaLM is the control for it, the other codon-resolution RNA encoder; ESM-2 150M checks whether the protein slot was simply too small. The remaining five swaps are one line away, with a disk projection printed before anything is extracted.

Two-swap triples, the plan's "promising triples", follow once the single swaps show which are promising.

---

## Setup

In [ ]:
# Standard library
import datetime
import json
import os
import time
from pathlib import Path

# Run any operation the Apple GPU backend lacks on the CPU; must precede importing torch
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Data handling
import numpy as np
import pandas as pd
import torch

# Shared project code in Code/shared_code; this notebook lives in Code/stage4
import sys
CODE_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "shared_code").is_dir()), Path.cwd() / "Code")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from shared_code import datasets, encoders
from shared_code.encoders import ENCODERS
from shared_code.fusion import blf, train
from shared_code.fusion import token_embeddings as tokens

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("torch", torch.__version__, "| Apple GPU available:", torch.backends.mps.is_available())

### Settings

Token caches for the swapped encoders go beside BioLangFusion's own, under `Code/stage3/embeddings/<benchmark>/<encoder>/`, written by the same resumable extraction as Stage 3 step 2. Training results go to `results/phaseB/<benchmark>/<encoder>/`, one JSON per run with its per-row test predictions.

Everything that is not the encoder stays at Phase A's setting: Equation 1's convolution, 100 TextCNN channels, the paper's recipe. The gated head's entropy weight is the one validation chose for the original encoders on the same benchmark, rather than re-selected for each swap; re-selecting would triple the gated runs.

In [ ]:
BENCHMARKS = ("cov_vac_degradation", "fungal_expression", "ecoli_proteins",
              "mrna_stability")                                 # smallest first, costliest last
CACHE_ROOT = Path("../stage3/embeddings")
RESULTS_ROOT = Path("results") / "phaseB"
ORIGINAL_RESULTS = {                                   # the original encoders' runs
    "cov_vac_degradation": [Path("results/phaseA/cov_vac_degradation")],
    "ecoli_proteins": [Path("results/phaseA/ecoli_proteins")],
    "fungal_expression": [Path("../stage3/results/fungal_expression"),
                          Path("results/phaseA/fungal_expression")],
    "mrna_stability": [Path("results/phaseA/mrna_stability")],
}
STAGE1_ALIGNMENT = Path("../../docs/figures/stability/results.json")

TCONV = "equation"
CHANNELS = 100
BATCH_SIZE = 32
MAX_EPOCHS = 500
SEED = 0
CHUNK_SIZE = tokens.DEFAULT_CHUNK
DTYPE = np.float16

SWAPS = [("dna", "nt500m"), ("dna", "hyenadna"),
         ("rna", "rinalmo"), ("rna", "mrnafm"), ("rna", "calm"),
         ("prot", "esm2_35m"), ("prot", "esm2_150m"), ("prot", "protbert")]
# mRNA stability is 23,929 sequences: all eight swaps would be about 100 GB of token cache.
# These three are the mRFP control's prediction and its two controls; widen the list here
# once the disk projection below says the rest fit.
STABILITY_SWAPS = [("rna", "mrnafm"), ("rna", "calm"), ("prot", "esm2_150m")]
ORIGINAL = dict(blf.ENCODER_OF_MODALITY)


def swaps_for(key):
    """The swaps run on one benchmark: all of them, except on the costly one."""
    return STABILITY_SWAPS if key == "mrna_stability" else SWAPS

DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
planned = sum(len(swaps_for(k)) for k in BENCHMARKS)
print(f"{planned} swap-benchmark pairs over {len(BENCHMARKS)} benchmarks, device {DEVICE}")
pd.DataFrame([{"slot": m, "replaces": ENCODERS[ORIGINAL[m]].label, "swapped in": ENCODERS[k].label,
               "grid": tokens.grid_resolution(k)} for m, k in SWAPS])

In [ ]:
# What the extraction will cost, before it starts. Tokens per sequence come from each
# encoder's own tokenization rule, so this is arithmetic rather than a guess, and the widths
# are the encoders' hidden sizes. Disk is the binding constraint on mRNA stability.
WIDTHS = {"nt500m": 1280, "hyenadna": 256, "rinalmo": 1280, "mrnafm": 1280, "calm": 768,
          "esm2_35m": 480, "esm2_150m": 640, "protbert": 1024}

projection = []
for key in BENCHMARKS:
    lengths = datasets.blf_benchmark_rows(key)["length"].to_numpy()
    for modality, encoder_key in swaps_for(key):
        per_row = np.array([tokens.tokens_for_length(ENCODERS[encoder_key], int(n))
                            for n in lengths])
        gigabytes = per_row.sum() * WIDTHS[encoder_key] * np.dtype(DTYPE).itemsize / 1e9
        projection.append({"benchmark": key, "encoder": ENCODERS[encoder_key].label,
                           "rows": len(lengths), "GB": round(gigabytes, 1)})
projection = pd.DataFrame(projection)
display(projection.pivot_table(index="benchmark", columns="encoder", values="GB",
                               aggfunc="sum").fillna(""))
print(f"total token cache for the planned swaps: {projection['GB'].sum():.0f} GB")
print("All of it stays out of Git, under Code/stage3/embeddings/<benchmark>/<encoder>/.")

### The original encoders' runs

Phase B is read against them, and the gated head's entropy weight comes from them, so they have to exist first: Phase A for CoV-Vac and E. coli, Stage 3 step 3 for fungal.

In [ ]:
def label(r):
    return "mil (lam 0)" if r.variant == "mil" and r.config["entropy_lambda"] == 0 else r.variant


def original_runs(key):
    """Seed-0 runs of BioLangFusion's own encoders at Phase A's settings.

    A unimodal model reads no grid, so its runs count under either convolution setting.
    """
    runs = []
    for folder in ORIGINAL_RESULTS[key]:
        for path in sorted(folder.glob("*.json")):
            r = train.Result.load(path)
            if r.config["channels"] != CHANNELS or r.config["seed"] != SEED:
                continue
            if r.variant.startswith("unimodal:") or r.config.get("tconv") == TCONV:
                runs.append(r)
    return runs


def original_lambda(key):
    gated = [r for r in original_runs(key) if r.variant == "mil"
             and r.config["entropy_lambda"] > 0 and r.val_score is not None]
    if not gated:
        raise RuntimeError(f"{key}: no gated-head runs for the original encoders yet; "
                           "run Phase A first")
    return max(gated, key=lambda r: r.val_score).config["entropy_lambda"]


status = []
for key in BENCHMARKS:
    runs = original_runs(key)
    have = sorted({label(r) for r in runs})
    try:
        lam = original_lambda(key)
    except RuntimeError:
        lam = None
    status.append({"benchmark": key, "original runs": len(runs), "variants": ", ".join(have),
                   "gated lambda": lam})
pd.DataFrame(status).set_index("benchmark")

---

## Token embeddings for the swapped encoders

First eight sequences through every swapped encoder, checking what the long run cannot afford to get wrong: each stream's token count is the one the codon grid expects. This loads every model once, which downloads it the first time.

In [ ]:
sample = datasets.blf_benchmark_rows("fungal_expression")["sequence"].tolist()[:8]
checks = []
for _, key in sorted({s for b in BENCHMARKS for s in swaps_for(b)}):
    encoder = ENCODERS[key]
    texts = tokens.inputs_for_rows(encoder, sample)
    tokenizer, model = encoders.load_encoder(encoder, DEVICE)
    try:
        model.eval()
        states = [tokens.token_states(t, tokenizer, model, encoder.max_len, DTYPE) for t in texts]
    finally:
        del model, tokenizer
        encoders.free_memory()
    want = [tokens.expected_tokens(encoder, s) for s in sample]
    got = [s.shape[0] for s in states]
    checks.append({"encoder": encoder.label, "width": states[0].shape[1],
                   "grid": tokens.grid_resolution(key), "tokens (first 3)": str(got[:3]),
                   "token counts match": "ok" if got == want else f"MISMATCH want {want[:3]}",
                   "finite": "ok" if all(np.isfinite(s).all() for s in states) else "NaN/inf"})
pd.DataFrame(checks).set_index("encoder")

The extraction itself. Resumable shard by shard, like Stage 3 step 2: rerun after an interruption and it continues where it stopped. The sizes it prints are the disk it uses; all of it stays out of Git.

In [ ]:
started = time.perf_counter()
written = []
for key in BENCHMARKS:
    rows = datasets.blf_benchmark_rows(key)
    print(f"=== {key}: {len(rows)} sequences ({datetime.datetime.now():%H:%M:%S}) ===")
    for _, encoder_key in swaps_for(key):
        encoder = ENCODERS[encoder_key]
        texts = tokens.inputs_for_rows(encoder, rows["sequence"].tolist())
        cache, computed = tokens.extract(encoder, texts, CACHE_ROOT / key / encoder_key, DEVICE,
                                         chunk_size=CHUNK_SIZE, dtype=DTYPE)
        written.append({"benchmark": key, "encoder": encoder.label, "sequences": len(cache),
                        "tokens": cache.meta["total_tokens"], "width": cache.width,
                        "GB": round(cache.meta["bytes"] / 1e9, 2),
                        "this run": "extracted" if computed else "already complete"})
        print(f"  {encoder.label}: {written[-1]['this run']}, {written[-1]['GB']} GB")
print(f"\nextraction finished in {(time.perf_counter() - started) / 60:.1f} minutes")
pd.DataFrame(written).set_index(["benchmark", "encoder"])

Each swapped triple opened the way training will open it: `BenchmarkStreams.open` checks that every cache holds one row per sequence with the token count that encoder's tokenization implies, so a cache from the wrong encoder or the wrong rows fails here rather than inside a run.

In [ ]:
STREAMS = {}
for key in BENCHMARKS:
    for modality, encoder_key in swaps_for(key):
        STREAMS[key, encoder_key] = blf.BenchmarkStreams.open(key, CACHE_ROOT, {modality: encoder_key})
pd.DataFrame([{"benchmark": key, "swapped in": enc, "encoders": s.encoders,
               "grid": s.resolutions, "widths": s.widths}
              for (key, enc), s in STREAMS.items()]).set_index(["benchmark", "swapped in"])

---

## The sweep

For every swap on every benchmark: the swapped-in encoder alone (and, when its stream is pooled or upsampled onto the grid, the same encoder read on the grid), then the three fusion heads with that encoder in place of the original. The cheap runs go first for every swap, cross-attention last, so an interrupted night still leaves every swap measured with two of the three heads. Resumable like Phase A. Stop the Mac sleeping first (`caffeinate -i`).

In [ ]:
def config(**overrides):
    return train.TrainConfig(batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS, channels=CHANNELS,
                             tconv=TCONV, seed=SEED, **overrides)


def swap_dir(key, encoder_key):
    return RESULTS_ROOT / key / encoder_key


def cheap_runs(key, modality, encoder_key):
    runs = [(f"unimodal:{modality}", {})]
    if STREAMS[key, encoder_key].resolutions[modality] != "codon":
        runs.append((f"unimodal-grid:{modality}", {}))
    return runs + [("concat", {}), ("mil", {"entropy_lambda": original_lambda(key)})]


started = time.perf_counter()
for key in BENCHMARKS:
    for modality, encoder_key in swaps_for(key):
        print(f"\n=== {key}: {ENCODERS[encoder_key].label} in the {modality} slot ===")
        for variant, overrides in cheap_runs(key, modality, encoder_key):
            train.run(STREAMS[key, encoder_key], variant, config(**overrides), DEVICE,
                      swap_dir(key, encoder_key))
for key in BENCHMARKS:
    for modality, encoder_key in swaps_for(key):
        print(f"\n=== {key}: {ENCODERS[encoder_key].label}, cross-attention ===")
        train.run(STREAMS[key, encoder_key], "cross", config(), DEVICE, swap_dir(key, encoder_key))
print(f"\nsweep finished in {(time.perf_counter() - started) / 3600:.2f} hours")

---

## What each swap does

For each benchmark, one row per triple: BioLangFusion's own first, then each swap. The singles are the three encoders of that triple, each alone; "best single" is the one validation picks among them (including a gridded reading where there is one), and Δfusion is each head minus that best single, the plan's measure of fusion benefit. "vs original" is the same head with the original encoders, on the same rows.

In [ ]:
SCALE = {"ecoli_proteins": 100.0}
HEADS = ("concat", "mil", "cross")


def swap_runs(key, encoder_key):
    return [train.Result.load(p) for p in sorted(swap_dir(key, encoder_key).glob("*.json"))]


def triple(key, modality=None, encoder_key=None):
    """{variant: Result} for one encoder triple: the original runs, with a swap's own runs
    replacing everything that involves the swapped slot."""
    lam = original_lambda(key)
    runs = {}
    for r in original_runs(key):
        name = label(r)
        if name == "mil" and r.config["entropy_lambda"] != lam:
            continue
        runs[name] = r
    if encoder_key is not None:
        for name in [n for n in runs if n in HEADS or n.endswith(f":{modality}")]:
            del runs[name]
        for r in swap_runs(key, encoder_key):
            runs[label(r)] = r
    return runs


def best_single(runs):
    singles = {n: r for n, r in runs.items() if n.startswith("unimodal") and r.val_score is not None}
    return max(singles, key=lambda n: singles[n].val_score) if singles else None


def summary(key):
    scale = SCALE.get(key, 1.0)
    original = triple(key)
    rows = []
    for modality, encoder_key in [(None, None), *swaps_for(key)]:
        runs = triple(key, modality, encoder_key)
        best = best_single(runs)
        row = {"slot": modality or "none",
               "encoder": ENCODERS[encoder_key].label if encoder_key else "BioLangFusion's own",
               "alone": runs.get(f"unimodal:{modality}").test_score * scale
               if modality and f"unimodal:{modality}" in runs else np.nan,
               "best single": best, "best single score": runs[best].test_score * scale if best else np.nan}
        for head in HEADS:
            score = runs[head].test_score * scale if head in runs else np.nan
            row[head] = score
            row[f"{head} Δfusion"] = score - row["best single score"]
            row[f"{head} vs original"] = (score - original[head].test_score * scale
                                          if head in original and encoder_key else np.nan)
        rows.append(row)
    return pd.DataFrame(rows).set_index(["slot", "encoder"])


for key in BENCHMARKS:
    print(f"\n{key}" + (" (accuracy x 100)" if key in SCALE else " (Spearman)"))
    try:
        display(summary(key).round(3))
    except (RuntimeError, KeyError) as missing:
        print("  not ready:", missing)

## What the test splits can resolve

The same paired bootstrap as Phase A, swap against original on identical rows. Two comparisons per head and swap: the head with the swap against the same head with the original encoders, which is the plan's success criterion; and the head with the swap against its own triple's best single encoder, which is Δfusion. With 400 to 712 test rows, the fungal result says to expect differences of 0.02 to stay unresolved.

In [ ]:
def resolve(key, draws=2000):
    streams = STREAMS[key, swaps_for(key)[0][1]]
    test = streams.split_rows("test")
    truth = streams.labels[test]
    groups = streams.rows["sequence"].to_numpy()[test]
    original = triple(key)
    predictions, comparisons = {}, []
    for name, r in original.items():
        predictions[f"original {name}"] = [np.asarray(r.test_prediction)]
    for modality, encoder_key in swaps_for(key):
        runs = triple(key, modality, encoder_key)
        best = best_single(runs)
        for name, r in runs.items():
            predictions[f"{encoder_key} {name}"] = [np.asarray(r.test_prediction)]
        for head in HEADS:
            if head not in runs or head not in original:
                continue
            comparisons.append((f"{encoder_key} {head} vs original",
                                f"{encoder_key} {head}", f"original {head}"))
            if best is not None:
                comparisons.append((f"{encoder_key} {head} vs best single ({best})",
                                    f"{encoder_key} {head}", f"{encoder_key} {best}"))
    out = pd.DataFrame(train.seed_averaged_bootstrap(streams.task, truth, predictions, comparisons,
                                                     groups=groups, draws=draws, seed=0))
    scale = SCALE.get(key, 1.0)
    for column in ("difference", "ci_low", "ci_high"):
        out[column] = out[column] * scale
    out["resolved"] = np.where((out["ci_low"] > 0) | (out["ci_high"] < 0), "yes", "no")
    return out.set_index("comparison")[["difference", "ci_low", "ci_high", "first_wins", "resolved"]]


RESOLVED = {}
for key in BENCHMARKS:
    try:
        RESOLVED[key] = resolve(key)
    except (RuntimeError, KeyError) as missing:
        print(f"{key}: not ready ({missing})")
        continue
    print(f"\n{key}")
    display(RESOLVED[key].round(3))

## Beside each swap: what Stage 1 measured about the pairs it creates

The Stage 2 decision is that alignment is reported beside every fusion result. A swap changes two of the three pairs in a triple; these are Stage 1's measurements of those two pairs, raw and after the composition control, from the mRNA stability sample (981 sequences, the only dataset where Stage 1 measured all twelve encoders together on coding sequences). They are measured on a different dataset from the one each swap is scored on, so they are the frozen-encoder prediction H2 is about, not a property of these benchmarks.

They are read from the Stage 1 capture in `docs/figures/stability/results.json`. Its raw CKA agrees with the saved Stage 1 notebook; some composition-controlled values do not (RNA-FM with RiNALMo is 0.486 here against 0.378 in the notebook). That is the open Stage 1 discrepancy recorded in the run records, so read the controlled column as provisional until it is resolved.

In [ ]:
stage1 = json.loads(STAGE1_ALIGNMENT.read_text())


def matrix(name):
    m = stage1["similarity"][name]
    return pd.DataFrame(m["data"], index=m["index"], columns=m["columns"])


cka, cka_controlled = matrix("Linear CKA"), matrix("Linear CKA after control")
alignment = []
for modality, encoder_key in sorted({s for b in BENCHMARKS for s in swaps_for(b)}):
    triple_keys = {**ORIGINAL, modality: encoder_key}
    for other in (m for m in ORIGINAL if m != modality):
        a, b = encoder_key, triple_keys[other]
        alignment.append({"swap": ENCODERS[encoder_key].label, "pair": f"{ENCODERS[a].label} + {ENCODERS[b].label}",
                          "CKA": cka.loc[a, b], "CKA after composition control": cka_controlled.loc[a, b]})
    alignment.append({"swap": ENCODERS[encoder_key].label,
                      "pair": f"{ENCODERS[encoder_key].label} vs the {ENCODERS[ORIGINAL[modality]].label} it replaces",
                      "CKA": cka.loc[encoder_key, ORIGINAL[modality]],
                      "CKA after composition control": cka_controlled.loc[encoder_key, ORIGINAL[modality]]})
pd.DataFrame(alignment).set_index(["swap", "pair"]).round(3)

## Reading

Written after the run, from the tables above. The plan's success criterion: at least one swap improves reproducible test performance under the unchanged architecture. A negative result is informative too: the original encoders may already be strong, or the alternatives may not transfer under the fixed head.

## Deviations recorded for this step

On top of Phase A's:

| # | Deviation | Why | Effect if wrong |
| --- | --- | --- | --- |
| 1 | One seed per swap | Seed spread on fungal was 0.002 to 0.008, inside every test split's bootstrap interval | A swap whose gain is near the seed spread needs seeds before it is claimed |
| 2 | The gated head's entropy weight is the original encoders' choice, not re-selected per swap | Re-selecting triples the gated runs | A swap could look worse with the gated head than it would at its own best weight |
| 3 | DNABERT-2 is not swapped in yet | Its byte-pair tokens have no fixed length, so the grid needs an adapter designed and held fixed first | Phase B is missing one of the plan's DNA candidates until then |
| 4 | RiNALMo and ESM-2 35M are added to the plan's candidates | Stage 1 found them redundant with the encoders they replace, which is a testable H2 prediction | None for the plan's candidates; two extra rows |
| 5 | Stage 1 alignment is from the mRNA stability sample | It is the only dataset where Stage 1 measured all twelve encoders on coding sequences | The values are a prediction from another dataset, not a property of each benchmark |
| 6 | Three swaps on mRNA stability, not eight | Eight would be about 100 GB of token cache for one benchmark; the three are the mRFP control's prediction (mRNA-FM), its codon-resolution control (CaLM), and a larger protein encoder | A swap that would have helped on stability is untested there until the list is widened |
| 7 | Two-swap triples come later | They depend on which single swaps help | Phase B is incomplete until they run |